# 🏪 Store Intelligence — Brigade Road, Bangalore
**Store ID:** ST1008 | **Date:** 10 April 2026 | **GPU Required:** T4

## Before running
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Upload your 5 video clips to Google Drive at:
   ```
   MyDrive/purplle-challenge/clips/   ← put all .mp4 files here
   ```
3. Run all cells top to bottom (**Runtime → Run all**)
4. Download `events.jsonl` from the last cell

## What this does
- Detects people in every frame using YOLOv8n
- Tracks individuals across frames with ByteTrack
- Emits ENTRY/EXIT/ZONE_ENTER/ZONE_EXIT/ZONE_DWELL/BILLING events
- Handles re-entry, staff exclusion, cross-camera deduplication
- Validates schema against all required fields
- Saves `events.jsonl` to your Google Drive automatically

In [ ]:
# ── CELL 1: Verify GPU ─────────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('✅ Ready to run detection')
else:
    print('❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── CELL 2: Install dependencies ────────────────────────────────────────────
%%capture
!pip install ultralytics==8.2.18 supervision==0.20.0 opencv-python-headless typer
print('✅ Dependencies installed')

In [ ]:
# ── CELL 3: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE = '/content/drive/MyDrive/purplle-challenge'
CLIPS_DIR  = f'{DRIVE_BASE}/clips'
OUTPUT     = f'{DRIVE_BASE}/events.jsonl'

# Verify clips exist
if not os.path.exists(CLIPS_DIR):
    os.makedirs(CLIPS_DIR, exist_ok=True)
    print(f'⚠️  Created {CLIPS_DIR} — upload your .mp4 files there now')
else:
    clips = [f for f in os.listdir(CLIPS_DIR) if f.endswith('.mp4')]
    print(f'✅ Found {len(clips)} clips:')
    for c in sorted(clips):
        size_mb = os.path.getsize(f'{CLIPS_DIR}/{c}') / 1e6
        print(f'   {c} ({size_mb:.0f} MB)')

In [ ]:
# ── CELL 4: Write pipeline code ─────────────────────────────────────────────
import os
os.makedirs('/content/pipeline', exist_ok=True)

# ── emit.py ──
with open('/content/pipeline/__init__.py', 'w') as f:
    f.write('')

with open('/content/pipeline/emit.py', 'w') as f:
    f.write('''
from __future__ import annotations
import json, os, uuid
from datetime import datetime, timezone
from typing import Optional

OUTPUT_PATH = os.getenv("EVENTS_OUTPUT_PATH", "/content/drive/MyDrive/purplle-challenge/events.jsonl")

def emit_event(*, store_id, camera_id, visitor_id, event_type, timestamp,
               zone_id=None, dwell_ms=0, is_staff=False, confidence,
               queue_depth=None, sku_zone=None, session_seq=None,
               is_partial_occlusion=None, reid_confidence=None):
    assert store_id and camera_id and visitor_id and event_type
    assert 0.0 <= confidence <= 1.0, f"bad conf {confidence}"
    zone_events = {"ZONE_ENTER","ZONE_EXIT","ZONE_DWELL",
                   "BILLING_QUEUE_JOIN","BILLING_QUEUE_ABANDON"}
    if event_type in zone_events:
        assert zone_id, f"zone_id required for {event_type}"
    if timestamp.tzinfo is None:
        timestamp = timestamp.replace(tzinfo=timezone.utc)
    event = {
        "event_id": str(uuid.uuid4()),
        "store_id": store_id,
        "camera_id": camera_id,
        "visitor_id": visitor_id,
        "event_type": event_type,
        "timestamp": timestamp.isoformat(),
        "zone_id": zone_id,
        "dwell_ms": dwell_ms,
        "is_staff": is_staff,
        "confidence": round(confidence, 4),
        "metadata": {
            "queue_depth": queue_depth,
            "sku_zone": sku_zone,
            "session_seq": session_seq,
            "is_partial_occlusion": is_partial_occlusion,
            "reid_confidence": reid_confidence,
        },
    }
    os.makedirs(os.path.dirname(os.path.abspath(OUTPUT_PATH)), exist_ok=True)
    with open(OUTPUT_PATH, "a") as f:
        f.write(json.dumps(event) + "\\n")
    return event

def make_visitor_id(track_id, store_id):
    short = hex(abs(hash(f"{store_id}_{track_id}")))[2:8]
    return f"VIS_{short}"
''')

# ── tracker.py ──
with open('/content/pipeline/tracker.py', 'w') as f:
    f.write('''
from __future__ import annotations
import os
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Optional
import numpy as np
import supervision as sv
import sys
sys.path.insert(0, "/content")
from pipeline.emit import emit_event, make_visitor_id

REENTRY_WINDOW_MINUTES = int(os.getenv("REENTRY_WINDOW_MINUTES", "30"))
TRIPWIRE_POSITION      = float(os.getenv("TRIPWIRE_POSITION", "0.65"))
PARTIAL_OCCLUSION_CONF = float(os.getenv("PARTIAL_OCCLUSION_CONF", "0.50"))
QUEUE_DEPTH_THRESHOLD  = int(os.getenv("QUEUE_DEPTH_THRESHOLD", "2"))
BILLING_KEYWORDS       = {"BILLING", "CHECKOUT", "CASHIER", "COUNTER", "CASH"}

def _is_billing(z): return any(k in z.upper() for k in BILLING_KEYWORDS)

class VisitorRegistry:
    CROSS_CAM_WINDOW = 30
    def __init__(self):
        self._entry_registry = []
        self._visitor_cameras = defaultdict(set)
        self._global_seen = {}
        self.exited = {}
        self._matched_to_floor = set()
    def register_entry(self, visitor_id, entry_time, camera_id):
        self._entry_registry.append((entry_time, visitor_id))
        self._visitor_cameras[visitor_id].add(camera_id)
        cutoff = entry_time - timedelta(seconds=self.CROSS_CAM_WINDOW * 2)
        self._entry_registry = [(t,v) for t,v in self._entry_registry if t >= cutoff]
    def lookup_for_floor(self, frame_time):
        cutoff = frame_time - timedelta(seconds=self.CROSS_CAM_WINDOW)
        candidates = [(t,v) for t,v in self._entry_registry
                      if t >= cutoff and v not in self._matched_to_floor]
        if not candidates: return None
        best = max(candidates, key=lambda x: x[0])[1]
        self._matched_to_floor.add(best)
        return best
    def mark_camera(self, visitor_id, camera_id, frame_time):
        self._visitor_cameras[visitor_id].add(camera_id)
        self._global_seen[visitor_id] = frame_time
    def camera_count(self, visitor_id):
        return len(self._visitor_cameras.get(visitor_id, set()))

@dataclass
class VisitorSession:
    visitor_id: str; track_id: int; store_id: str
    first_seen: datetime; last_seen: datetime
    zone_history: list = field(default_factory=list)
    camera_ids_seen: set = field(default_factory=set)
    crossed_tripwire: bool = False; exited: bool = False
    is_staff: bool = False; session_seq: int = 0
    in_billing_zone: bool = False
    billing_entry_time: Optional[datetime] = None

class StoreTracker:
    def __init__(self, store_id, camera_id, W, H, registry=None):
        self.store_id = store_id; self.camera_id = camera_id
        self.frame_width = W; self.frame_height = H
        self.registry = registry or VisitorRegistry()
        self.tracker = sv.ByteTrack()
        self.tripwire_y = int(H * TRIPWIRE_POSITION)
        self._prev_centroids = {}; self._sessions = {}
        self._track_to_visitor = {}; self._billing_occupants = set()

    def update(self, detections, frame_time, zone_map=None):
        events = []
        if len(detections) == 0: return events
        tracked = self.tracker.update_with_detections(detections)
        current_billing = set()
        for i in range(len(tracked)):
            tid = int(tracked.tracker_id[i])
            bbox = tracked.xyxy[i]
            conf = float(tracked.confidence[i]) if tracked.confidence is not None else 0.5
            cx = (bbox[0]+bbox[2])/2; cy = (bbox[1]+bbox[3])/2
            is_partial = conf < PARTIAL_OCCLUSION_CONF
            vid = self._get_or_create(tid, frame_time)
            sess = self._sessions[vid]
            sess.is_staff = self.registry.camera_count(vid) >= 3
            sess.last_seen = frame_time
            sess.camera_ids_seen.add(self.camera_id)
            self.registry.mark_camera(vid, self.camera_id, frame_time)
            if "ENTRY" in self.camera_id.upper():
                events.extend(self._tripwire(tid, vid, cx, cy, conf, sess.is_staff, frame_time, sess, is_partial))
            if zone_map:
                zid = self._bbox_to_zone(bbox, zone_map)
                if zid:
                    if _is_billing(zid): current_billing.add(vid)
                    events.extend(self._handle_zone(vid, zid, conf, sess.is_staff, frame_time, sess, is_partial))
            self._prev_centroids[tid] = cy
        events.extend(self._billing_queue(current_billing, frame_time, zone_map))
        return events

    def _get_or_create(self, tid, frame_time):
        if tid in self._track_to_visitor: return self._track_to_visitor[tid]
        is_floor = "FLOOR" in self.camera_id.upper() or "BILLING" in self.camera_id.upper()
        if is_floor:
            matched = self.registry.lookup_for_floor(frame_time)
            if matched and matched not in self._track_to_visitor.values():
                self._track_to_visitor[tid] = matched
                if matched not in self._sessions:
                    self._sessions[matched] = VisitorSession(matched, tid, self.store_id, frame_time, frame_time)
                return matched
            vid = make_visitor_id(tid, f"{self.store_id}_{self.camera_id}")
        else:
            vid = make_visitor_id(tid, self.store_id)
        if vid in self.registry.exited:
            exit_t = self.registry.exited[vid]
            if frame_time - exit_t > timedelta(minutes=REENTRY_WINDOW_MINUTES):
                del self.registry.exited[vid]; vid = f"{vid}_r"
            else:
                if vid in self._sessions: self._sessions[vid].exited = False
        if vid not in self._sessions:
            self._sessions[vid] = VisitorSession(vid, tid, self.store_id, frame_time, frame_time)
        self._track_to_visitor[tid] = vid
        return vid

    def _tripwire(self, tid, vid, cx, cy, conf, is_staff, ft, sess, partial):
        events = []
        prev_y = self._prev_centroids.get(tid)
        if prev_y is None: return events
        down = prev_y < self.tripwire_y <= cy
        up   = prev_y > self.tripwire_y >= cy
        if down and not sess.crossed_tripwire:
            sess.crossed_tripwire = True; sess.session_seq += 1
            is_re = vid in self.registry.exited
            events.append(emit_event(store_id=self.store_id, camera_id=self.camera_id,
                visitor_id=vid, event_type="REENTRY" if is_re else "ENTRY",
                timestamp=ft, confidence=conf, is_staff=is_staff,
                session_seq=sess.session_seq, is_partial_occlusion=partial))
            self.registry.register_entry(vid, ft, self.camera_id)
            if is_re: del self.registry.exited[vid]
        elif up and sess.crossed_tripwire and not sess.exited:
            sess.exited = True; sess.session_seq += 1
            self.registry.exited[vid] = ft
            events.append(emit_event(store_id=self.store_id, camera_id=self.camera_id,
                visitor_id=vid, event_type="EXIT", timestamp=ft,
                confidence=conf, is_staff=is_staff, session_seq=sess.session_seq,
                is_partial_occlusion=partial))
        return events

    def _handle_zone(self, vid, zone_id, conf, is_staff, ft, sess, partial):
        events = []
        last = sess.zone_history[-1] if sess.zone_history else None
        if zone_id == last: return events
        if last is not None:
            sess.session_seq += 1
            events.append(emit_event(store_id=self.store_id, camera_id=self.camera_id,
                visitor_id=vid, event_type="ZONE_EXIT", timestamp=ft, zone_id=last,
                confidence=conf, is_staff=is_staff, session_seq=sess.session_seq,
                sku_zone=last, is_partial_occlusion=partial))
        sess.zone_history.append(zone_id); sess.session_seq += 1
        events.append(emit_event(store_id=self.store_id, camera_id=self.camera_id,
            visitor_id=vid, event_type="ZONE_ENTER", timestamp=ft, zone_id=zone_id,
            confidence=conf, is_staff=is_staff, session_seq=sess.session_seq,
            sku_zone=zone_id, is_partial_occlusion=partial))
        return events

    def _billing_queue(self, current, ft, zone_map):
        events = []
        billing_zone_id = "BILLING"
        if zone_map:
            for zid in zone_map:
                if _is_billing(zid): billing_zone_id = zid; break
        newly = current - self._billing_occupants
        depth = len(self._billing_occupants)
        for vid in newly:
            if depth >= QUEUE_DEPTH_THRESHOLD:
                sess = self._sessions.get(vid)
                if not sess: continue
                sess.in_billing_zone = True; sess.billing_entry_time = ft
                sess.session_seq += 1
                events.append(emit_event(store_id=self.store_id, camera_id=self.camera_id,
                    visitor_id=vid, event_type="BILLING_QUEUE_JOIN", timestamp=ft,
                    zone_id=billing_zone_id, confidence=0.85, is_staff=sess.is_staff,
                    session_seq=sess.session_seq, queue_depth=depth, sku_zone=billing_zone_id))
        self._billing_occupants = current
        return events

    def _bbox_to_zone(self, bbox, zone_map):
        cx = (bbox[0]+bbox[2])/2; cy = (bbox[1]+bbox[3])/2
        for zid, info in zone_map.items():
            if self._pip(cx, cy, info.get("polygon",[])): return zid
        return None

    @staticmethod
    def _pip(x, y, poly):
        n, inside, j = len(poly), False, len(poly)-1
        for i in range(n):
            xi,yi = poly[i]; xj,yj = poly[j]
            if ((yi>y)!=(yj>y)) and (x < (xj-xi)*(y-yi)/(yj-yi)+xi):
                inside = not inside
            j = i
        return inside
''')

print('✅ Pipeline code written to /content/pipeline/')

In [ ]:
# ── CELL 5: Write store_layout.json ─────────────────────────────────────────
import json, os

STORE_LAYOUT = {
  "ST1008": {
    "store_name": "Brigade_Bangalore",
    "store_id": "ST1008",
    "city": "Bangalore",
    "open_hours": {"open": "10:00", "close": "22:00"},
    "entry_camera": "CAM_ENTRY_01",
    "zones": {
      "BACK_WALL_LEFT":    {"display_name": "EB Korean / Face Shop / Good Vibes",
                           "brands": ["EB Korean","The Face Shop","Good Vibes"],
                           "department": "skin",
                           "polygon": [[0,0],[350,0],[350,200],[0,200]],
                           "camera": "CAM_FLOOR_01"},
      "BACK_WALL_CENTRE":  {"display_name": "DermDoc / Minimalist / Aqualogica",
                           "brands": ["DermDoc","Minimalist","Aqualogica"],
                           "department": "skin",
                           "polygon": [[350,0],[650,0],[650,200],[350,200]],
                           "camera": "CAM_FLOOR_01"},
      "BACK_WALL_RIGHT":   {"display_name": "Lakme Skin / Accessories",
                           "brands": ["Lakme Skin","Accessories"],
                           "department": "skin",
                           "polygon": [[650,0],[1000,0],[1000,200],[650,200]],
                           "camera": "CAM_FLOOR_01"},
      "FRAGRANCE":         {"display_name": "Fragrance",
                           "brands": ["Fragrance"],
                           "department": "fragrance",
                           "polygon": [[200,250],[350,250],[350,600],[200,600]],
                           "camera": "CAM_FLOOR_01"},
      "NAIL_UNIT":         {"display_name": "Nail Unit",
                           "brands": ["Nail Unit"],
                           "department": "personal-care",
                           "polygon": [[350,250],[450,250],[450,600],[350,600]],
                           "camera": "CAM_FLOOR_01"},
      "FOH":               {"display_name": "Front of House",
                           "brands": [],
                           "department": "general",
                           "polygon": [[450,200],[750,200],[750,650],[450,650]],
                           "camera": "CAM_FLOOR_01"},
      "MAKEUP_UNIT":       {"display_name": "Makeup Unit",
                           "brands": [],
                           "department": "makeup",
                           "polygon": [[500,350],[750,350],[750,550],[500,550]],
                           "camera": "CAM_FLOOR_01"},
      "BILLING":           {"display_name": "Cash Counter / Billing",
                           "brands": [],
                           "department": "billing",
                           "polygon": [[800,100],[1000,100],[1000,650],[800,650]],
                           "camera": "CAM_BILLING_01",
                           "is_billing": True},
      "PMU":               {"display_name": "PMU Zone",
                           "brands": ["PMU"],
                           "department": "personal-care",
                           "polygon": [[850,550],[1000,550],[1000,800],[850,800]],
                           "camera": "CAM_BILLING_01"},
      "BOTTOM_WALL_LEFT":  {"display_name": "Maybelline / Faces Canada / Lakme",
                           "brands": ["Maybelline","Faces Canada","Lakme"],
                           "department": "makeup",
                           "polygon": [[0,700],[350,700],[350,900],[0,900]],
                           "camera": "CAM_FLOOR_01"},
      "BOTTOM_WALL_CENTRE":{"display_name": "Colorbar / Sugar / Swiss Beauty / Renee NY Bae",
                           "brands": ["Colorbar","Sugar","Swiss Beauty","Renee NY Bae"],
                           "department": "makeup",
                           "polygon": [[350,700],[650,700],[650,900],[350,900]],
                           "camera": "CAM_FLOOR_01"},
      "BOTTOM_WALL_RIGHT": {"display_name": "Alps Goodness / Streax",
                           "brands": ["Alps Goodness","Streax"],
                           "department": "hair",
                           "polygon": [[650,700],[850,700],[850,900],[650,900]],
                           "camera": "CAM_FLOOR_01"}
    }
  }
}

LAYOUT_PATH = '/content/store_layout.json'
with open(LAYOUT_PATH, 'w') as f:
    json.dump(STORE_LAYOUT, f, indent=2)
print(f'✅ store_layout.json written — {len(STORE_LAYOUT["ST1008"]["zones"])} zones')

In [ ]:
# ── CELL 6: CONFIGURE AND RUN DETECTION ─────────────────────────────────────
# ⚠️  Edit these values to match your video clips
# ─────────────────────────────────────────────────────────────────────────────
STORE_ID       = 'ST1008'
CLIP_START     = '2026-04-10T12:00:00Z'   # ISO-8601 UTC — must match POS data date
YOLO_MODEL     = 'yolov8n.pt'             # yolov8n=fastest, yolov8s=more accurate
CONFIDENCE     = 0.30                     # detection threshold — keep at 0.30
SAMPLE_EVERY_N = 3                        # 1=every frame, 3=5fps effective (3x faster)
# ─────────────────────────────────────────────────────────────────────────────

import sys, json, os, time
from datetime import datetime, timezone, timedelta
from pathlib import Path
import cv2
import supervision as sv
from ultralytics import YOLO

sys.path.insert(0, '/content')
from pipeline.tracker import StoreTracker, VisitorRegistry
from pipeline.emit import emit_event

os.environ['EVENTS_OUTPUT_PATH'] = OUTPUT

# Clear previous output if exists
if os.path.exists(OUTPUT):
    os.remove(OUTPUT)
    print(f'Cleared previous {OUTPUT}')

# Load model
print(f'Loading {YOLO_MODEL}...')
model = YOLO(YOLO_MODEL)
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f'Running on: {device.upper()}')

# Load layout
with open(LAYOUT_PATH) as f:
    layout = json.load(f)
store_layout = layout.get(STORE_ID, layout)
zone_map = store_layout.get('zones', {})
print(f'Zones: {list(zone_map.keys())}')

def frame_to_ts(idx, start, fps=15):
    return start + timedelta(seconds=idx / fps)

def get_clips(clips_dir, store_id):
    clips = {}
    for f in sorted(Path(clips_dir).glob('*.mp4')):
        stem = f.stem.upper()
        if store_id.upper() in stem:
            name = f.stem.replace(store_id+'_','').replace(store_id,'')
            cam = name if name.upper().startswith('CAM_') else f'CAM_{name.upper()}'
        elif 'ENTRY' in stem:
            cam = 'CAM_ENTRY_01'
        elif 'BILLING' in stem or 'CASH' in stem:
            cam = 'CAM_BILLING_01'
        elif 'FLOOR' in stem:
            cam = 'CAM_FLOOR_01'
        elif f.stem in ('1','01'):
            cam = 'CAM_ENTRY_01'
        elif f.stem in ('3','03'):
            cam = 'CAM_BILLING_01'
        else:
            cam = 'CAM_FLOOR_01'
        if cam in clips:
            cam = f'CAM_{f.stem.upper()}'
        clips[cam] = str(f)
    # Sort: ENTRY first
    order = lambda c: 0 if 'ENTRY' in c else (2 if 'BILLING' in c else 1)
    return dict(sorted(clips.items(), key=lambda x: order(x[0])))

clips = get_clips(CLIPS_DIR, STORE_ID)
print(f'\nClips found ({len(clips)}):')
for cam, path in clips.items():
    print(f'  {cam}: {os.path.basename(path)}')

if not clips:
    print(f'\n❌ No .mp4 files found in {CLIPS_DIR}')
    print('Upload your videos to that folder in Google Drive and re-run this cell.')
else:
    start_dt = datetime.fromisoformat(CLIP_START.replace('Z','+00:00'))
    registry = VisitorRegistry()
    grand_total = 0
    all_billing_exits = []

    for camera_id, clip_path in clips.items():
        cap = cv2.VideoCapture(clip_path)
        W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f'\n── {camera_id}: {os.path.basename(clip_path)} ({total} frames, {W}x{H}) ──')

        tracker = StoreTracker(STORE_ID, camera_id, W, H, registry=registry)
        active_zone_map = zone_map if ('FLOOR' in camera_id or 'BILLING' in camera_id) else None
        dwell_tracker = {}
        ev_count = 0; idx = 0; t0 = time.time()

        while True:
            ret, frame = cap.read()
            if not ret: break
            if idx % SAMPLE_EVERY_N != 0:
                idx += 1; continue

            ft = frame_to_ts(idx, start_dt)
            res = model(frame, classes=[0], conf=CONFIDENCE, device=device, verbose=False)[0]
            dets = sv.Detections.from_ultralytics(res)
            events = tracker.update(dets, ft, active_zone_map)
            ev_count += len(events)

            # ZONE_DWELL
            for vid, sess in tracker._sessions.items():
                if not sess.zone_history: continue
                zid = sess.zone_history[-1]
                dwell_tracker.setdefault(vid, {})
                last = dwell_tracker[vid].get(zid)
                if last is None:
                    dwell_tracker[vid][zid] = ft
                elif (ft - last).total_seconds() >= 30:
                    dwell_ms = int((ft - last).total_seconds() * 1000)
                    sess.session_seq += 1
                    emit_event(store_id=STORE_ID, camera_id=camera_id, visitor_id=vid,
                               event_type='ZONE_DWELL', timestamp=ft, zone_id=zid,
                               dwell_ms=dwell_ms, is_staff=sess.is_staff, confidence=0.9,
                               session_seq=sess.session_seq, sku_zone=zid)
                    dwell_tracker[vid][zid] = ft; ev_count += 1

            if idx % 300 == 0 and idx > 0:
                pct = idx/total*100; elapsed = time.time()-t0
                fps_a = idx/elapsed if elapsed>0 else 0
                eta = (total-idx)/(fps_a*SAMPLE_EVERY_N+0.001)
                print(f'  {pct:.0f}% — {ev_count} events — {fps_a:.1f} fps — ETA {eta/60:.1f} min')
            idx += 1

        cap.release()
        elapsed = time.time()-t0
        print(f'  ✓ Done: {ev_count} events in {elapsed/60:.1f} min')
        grand_total += ev_count

    # Billing abandon post-processing
    print(f'\nRunning abandon detection...')
    # Load POS timestamps
    import csv
    POS_CSV = '/content/pos_transactions.csv'
    pos_timestamps = []
    if os.path.exists(POS_CSV):
        with open(POS_CSV) as f:
            for row in csv.DictReader(f):
                if row['store_id'].strip() == STORE_ID:
                    ts = datetime.fromisoformat(row['timestamp'].replace('Z','+00:00')).replace(tzinfo=None)
                    pos_timestamps.append(ts)
    abandons = 0
    for rec in all_billing_exits:
        if rec.get('is_staff'): continue
        exit_t = rec['exit_time'].replace(tzinfo=None)
        window_end = exit_t + timedelta(minutes=5)
        if not any(exit_t <= ts <= window_end for ts in pos_timestamps):
            emit_event(store_id=STORE_ID, camera_id=rec['camera_id'],
                       visitor_id=rec['visitor_id'], event_type='BILLING_QUEUE_ABANDON',
                       timestamp=rec['exit_time'], zone_id='BILLING',
                       confidence=0.80, is_staff=False)
            abandons += 1
    grand_total += abandons

    print(f'  BILLING_QUEUE_ABANDON: {abandons}')
    print(f'\n✅ DONE: {grand_total} total events → {OUTPUT}')

In [ ]:
# ── CELL 7: Write pos_transactions.csv to Colab ─────────────────────────────
# This is the real Brigade Road POS data from 10 April 2026
pos_data = """store_id,transaction_id,timestamp,basket_value_inr
ST1008,TXN_104338647,2026-04-10T12:15:05Z,1647.0
ST1008,TXN_104341290,2026-04-10T12:42:18Z,11367.0
ST1008,TXN_104346717,2026-04-10T13:41:55Z,396.0
ST1008,TXN_104347785,2026-04-10T13:55:16Z,201.0
ST1008,TXN_104350137,2026-04-10T14:23:21Z,225.0
ST1008,TXN_104353598,2026-04-10T15:02:20Z,1165.0
ST1008,TXN_104357849,2026-04-10T15:46:39Z,599.0
ST1008,TXN_104358212,2026-04-10T15:50:44Z,402.0
ST1008,TXN_104359750,2026-04-10T16:08:03Z,799.0
ST1008,TXN_104362899,2026-04-10T16:45:32Z,4029.0
ST1008,TXN_104363838,2026-04-10T16:55:36Z,2713.0
ST1008,TXN_104368521,2026-04-10T17:44:44Z,299.0
ST1008,TXN_104369411,2026-04-10T17:55:02Z,2074.0
ST1008,TXN_104369867,2026-04-10T18:00:18Z,149.0
ST1008,TXN_104370397,2026-04-10T18:07:14Z,1155.0
ST1008,TXN_104373042,2026-04-10T18:41:51Z,2658.0
ST1008,TXN_104375288,2026-04-10T19:02:09Z,3441.0
ST1008,TXN_104377545,2026-04-10T19:21:55Z,4385.0
ST1008,TXN_104378732,2026-04-10T19:33:52Z,1198.0
ST1008,TXN_104379480,2026-04-10T19:41:29Z,2296.0
ST1008,TXN_104380754,2026-04-10T19:54:02Z,1749.0
ST1008,TXN_104383803,2026-04-10T20:25:04Z,1199.0
ST1008,TXN_104389493,2026-04-10T21:16:15Z,299.0
ST1008,TXN_104391745,2026-04-10T21:39:55Z,475.0
"""
with open('/content/pos_transactions.csv', 'w') as f:
    f.write(pos_data)
print('✅ pos_transactions.csv written (24 real transactions, 10 Apr 2026)')

In [ ]:
# ── CELL 8: Validate events.jsonl ───────────────────────────────────────────
import json
from collections import Counter

REQUIRED = {'event_id','store_id','camera_id','visitor_id','event_type',
            'timestamp','zone_id','dwell_ms','is_staff','confidence','metadata'}
VALID_TYPES = {'ENTRY','EXIT','ZONE_ENTER','ZONE_EXIT','ZONE_DWELL',
               'BILLING_QUEUE_JOIN','BILLING_QUEUE_ABANDON','REENTRY'}

events = []
with open(OUTPUT) as f:
    for i, line in enumerate(f):
        line = line.strip()
        if line:
            try: events.append(json.loads(line))
            except Exception as e: print(f'Line {i}: parse error {e}')

print(f'Total events: {len(events)}')

errors = []
ids = set()
for e in events:
    miss = REQUIRED - e.keys()
    if miss: errors.append(f'Missing keys {miss} in {e.get("event_id","?")}')
    if e.get('event_id') in ids: errors.append(f'Duplicate event_id {e["event_id"]}')
    ids.add(e.get('event_id'))
    if e.get('event_type') not in VALID_TYPES: errors.append(f'Bad type {e.get("event_type")}')
    if not (0 <= (e.get('confidence') or -1) <= 1): errors.append(f'Bad conf {e.get("confidence")}')

if errors:
    print(f'⚠️  {len(errors)} errors:')
    for err in errors[:10]: print(f'  {err}')
else:
    print('✅ All events pass schema validation')

types   = Counter(e['event_type'] for e in events)
cameras = Counter(e['camera_id'] for e in events)
n_vis   = len({e['visitor_id'] for e in events})
n_staff = sum(1 for e in events if e.get('is_staff'))

print(f'\n📊 Summary:')
print(f'  Unique visitors : {n_vis}')
print(f'  Staff events    : {n_staff}')
print(f'  Event types     :')
for t,n in sorted(types.items()): print(f'    {t}: {n}')
print(f'  By camera       :')
for c,n in sorted(cameras.items()): print(f'    {c}: {n}')

In [ ]:
# ── CELL 9: Download events.jsonl ────────────────────────────────────────────
print(f'events.jsonl is saved at: {OUTPUT}')
print('It is already in your Google Drive.')
print('\nTo download directly to your browser:')

from google.colab import files
files.download(OUTPUT)

print('\n📋 Next steps on your local machine:')
print('  1. Move events.jsonl to store-intelligence/data/events.jsonl')
print('  2. docker compose up --build -d')
print('  3. python pipeline/ingest_events.py --events ./data/events.jsonl')
print('  4. curl http://localhost:8000/stores/ST1008/metrics')